In [ ]:
!pip install -q "numpy<2.0.0"
import os
os.kill(os.getpid(), 9)

In [1]:
!pip install -q -U datasets

from datasets import load_dataset
import pandas as pd

raid_stream = load_dataset("liamdugan/raid", split="train", streaming=True)
data_sample = list(raid_stream.take(50000))
raid_df = pd.DataFrame(data_sample)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.7 MB/s eta 0:00:00


In [2]:
raid_df.shape

(50000, 11)

In [3]:
print((raid_df['model'] == 'human').sum())

4025


In [4]:
hc3 = load_dataset("json", data_files="hf://datasets/Hello-SimpleAI/HC3/all.jsonl")

rows = []
for i, row in enumerate(hc3["train"]):
    for ans in row["human_answers"]:
        rows.append({"text": ans, "label": "human", "source_id": f"hc3_{i}"})
    for ans in row["chatgpt_answers"]:
        rows.append({"text": ans, "label": "ai", "source_id": f"hc3_{i}"})

hc3_df = pd.DataFrame(rows)

all.jsonl: reconstructing file:   0%|          |  0.00B / 73.7MB            

all.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
hc3_df['label'].value_counts()

,count
label,
human,58546
ai,26903


In [6]:
raid_df['text'] = raid_df['generation']
raid_df['label'] = raid_df['model'].apply(lambda x: 'human' if x == 'human' else 'ai')
raid_df['source_id'] = raid_df['source_id'].apply(lambda x: f"raid_{x}")

combined_df = pd.concat(
    [raid_df[['text', 'label', 'source_id']], hc3_df[['text', 'label', 'source_id']]],
    ignore_index=True
)

In [7]:
combined_df['label'].value_counts()

,count
label,
ai,72878
human,62571


In [10]:
human_df = combined_df[combined_df['label'] == 'human']
ai_df = combined_df[combined_df['label'] == 'ai']

n_ai_needed = int(len(human_df) * 55 / 45)
ai_df_balanced = ai_df.sample(n=n_ai_needed, random_state=42, replace=True)

df = pd.concat([human_df, ai_df_balanced])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df['label'].value_counts(normalize=True))
print(df.shape)

label
ai       0.549998
human    0.450002
Name: proportion, dtype: float64
(139046, 3)


In [11]:
df.head(10)

,text,label,source_id
0,> Image segmentation is a process of partition...,ai,raid_b92ae04c-342d-4221-8ffc-cf81dae44f5b
1,"In this paper, we propose a novel approach to ...",ai,raid_60d8547e-cb69-417e-abc9-e54cc3d43d59
2,"Train horns are loud for a few reasons. First,...",ai,hc3_6081
3,Cocaine is made from coca leaves . Opiates ( s...,human,hc3_6207
4,It has been said that Rugby was invented durin...,human,hc3_9004
5,The task addressed by this paper concerns an a...,ai,raid_82d8563c-f93b-4006-baec-eea20df11b5f
6,"That interest rate (13%) is steep, and the bal...",human,hc3_22214
7,"Last names, also known as family names or surn...",ai,hc3_3845
8,"At a certain pressure , the nitrogen in the ai...",human,hc3_16121
9,Twitching is a common phenomenon that can affe...,ai,hc3_5527


In [12]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['source_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

x_train, y_train = train_df['text'], (train_df['label'] != 'human').astype(int)
x_test, y_test = test_df['text'], (test_df['label'] != 'human').astype(int)

In [13]:
x_train.head(5)

,text
0,> Image segmentation is a process of partition...
1,"In this paper, we propose a novel approach to ..."
2,"Train horns are loud for a few reasons. First,..."
3,Cocaine is made from coca leaves . Opiates ( s...
4,It has been said that Rugby was invented durin...


In [14]:
y_train.head(10)

,label
0,1
1,1
2,1
3,0
4,0
5,1
6,0
7,1
8,0
10,1


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

In [16]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB

params = [
    ('lr', LogisticRegression(C=1.0, max_iter=500)),
    ('sgd', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=1000, random_state=42)),
    ('nb', MultinomialNB(alpha=0.1))
]

In [17]:
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier

stack = StackingClassifier(
    estimators=params,
    final_estimator=XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        eval_metric='logloss'
    ),
    cv=3,
    n_jobs=-1
)

In [18]:
from sklearn.pipeline import Pipeline
model_pipeline = Pipeline([
    ('tf', tf),
    ('stacking', stack)
])

In [19]:
model_pipeline.fit(x_train, y_train)
y_pred = model_pipeline.predict(x_test)

In [20]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.9477375348256316
[[ 8864   488]
 [  600 10866]]
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      9352
           1       0.96      0.95      0.95     11466

    accuracy                           0.95     20818
   macro avg       0.95      0.95      0.95     20818
weighted avg       0.95      0.95      0.95     20818



In [21]:
samples = [
    "Artificial intelligence sucks and fucks human being's future",
    "Artificial intelligence has become an integral part of modern life.",
]
print(model_pipeline.predict(samples))
print(model_pipeline.predict_proba(samples))

[0 0]
[[0.7736324  0.2263676 ]
 [0.9577735  0.04222648]]


In [22]:
wrong = x_test[y_test != y_pred]
print(wrong.head(10))

748     http://papers.nips.cc/paper/4872-universal-sem...
1079    > For every n∈ω, we call an algebra {𝔾n;0≤i<j....
1185    The names of the months were originally based ...
1309    " Human sacrifices were once made on the hills...
1694    > We propose a novel image segmentation framew...
1768    The tax treatment of capital gains depends on ...
1842    Units are what we use to indicate what the kin...
1994    "Auld Lang Syne" is a song about saying goodby...
2352    In computing and computer science, a processor...
2458    When someone testifies for immunity, it means ...
Name: text, dtype: object


In [23]:
import pickle
pickle.dump(model_pipeline, open('aidetector.pkl', 'wb'))